# Chapter 8: Fourier Series and Periodic Forcing
## Complete Instructor Interactive Notebook

This supplementary chapter introduces Fourier series in a way that connects directly to differential equations.

The main idea is simple:

$$
\boxed{
\text{complicated periodic function}
\approx
\text{sum of sines and cosines}.
}
$$

Once a periodic forcing function is decomposed into harmonics, a linear differential equation can respond to each harmonic separately. Superposition then allows us to add the responses.

This chapter is organized into three sections:

1. **Section 8.1 — Fourier Series from Orthogonality**
2. **Section 8.2 — Symmetry, Piecewise Functions, Convergence, and Gibbs Phenomenon**
3. **Section 8.3 — Fourier Series as Periodic Forcing in Differential Equations**

The emphasis is on:

- understanding where the coefficient formulas come from;
- computing coefficients carefully;
- using symmetry to reduce work;
- interpreting partial sums graphically;
- understanding what happens at jump discontinuities;
- connecting Fourier series to forced oscillations and resonance.

In [ ]:
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing package(s):", ", ".join(_missing))
else:
    print("Notebook environment is ready.")

import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML, Markdown

try:
    import ipywidgets as widgets
    from ipywidgets import IntSlider, FloatSlider, Dropdown, interact
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

    def IntSlider(*args, value=0, **kwargs):
        return value

    def FloatSlider(*args, value=0.0, **kwargs):
        return value

    def Dropdown(*args, value=None, options=None, **kwargs):
        if value is not None:
            return value
        return options[0] if options else None

    def interact(function, **kwargs):
        values = {key: getattr(obj, "value", obj) for key, obj in kwargs.items()}
        function(**values)
        return None

plt.rcParams["figure.figsize"] = (9, 5.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

PI = np.pi

def square_wave(x):
    y = np.where(np.sin(x) >= 0, 1.0, -1.0)
    return y

def sawtooth_periodic(x):
    # 2pi-periodic extension of f(x)=x on (-pi,pi)
    return ((x + PI) % (2*PI)) - PI

def square_partial_sum(x, N):
    # N = number of odd harmonics retained
    s = np.zeros_like(np.asarray(x, dtype=float))
    for j in range(N):
        n = 2*j + 1
        s += 4/(PI*n) * np.sin(n*x)
    return s

def sawtooth_partial_sum(x, N):
    s = np.zeros_like(np.asarray(x, dtype=float))
    for n in range(1, N+1):
        s += 2*((-1)**(n+1))/n * np.sin(n*x)
    return s

def absx_partial_sum(x, N):
    # |x| = pi/2 - (4/pi) sum_{n odd} cos(nx)/n^2
    s = (PI/2)*np.ones_like(np.asarray(x, dtype=float))
    count = 0
    n = 1
    while count < N:
        s -= 4/(PI*n**2)*np.cos(n*x)
        n += 2
        count += 1
    return s

def display_animation(anim):
    try:
        return HTML(anim.to_jshtml())
    except Exception:
        return HTML(anim.to_html5_video())

# Section 8.1: Fourier Series from Orthogonality

## 1. Why Fourier Series?

Many physical inputs are periodic but are not sinusoidal.

Examples include:

- a square-wave voltage;
- a periodic on/off control signal;
- repeated impacts;
- seasonal forcing;
- a sawtooth signal;
- a periodically switched force.

A linear differential equation is especially easy to analyze when the forcing is a sine or cosine. Fourier's idea is therefore to represent a general periodic function as a sum of sinusoidal components.

For a $2\pi$-periodic function, we seek

$$
\boxed{
f(x)
\sim
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
\left[
a_n\cos(nx)+b_n\sin(nx)
\right].
}
$$

The symbol

$$
\sim
$$

means that the right-hand side is the Fourier series associated with $f$. Under suitable conditions, its partial sums converge to $f$ at points of continuity and to a midpoint value at jump discontinuities.

## 2. Periodic Functions

A function $f$ is periodic with period $T>0$ if

$$
\boxed{
f(x+T)=f(x)
}
$$

for every $x$ where the function is defined.

The smallest positive period, when it exists, is called the **fundamental period**.

For example,

$$
\sin x
$$

and

$$
\cos x
$$

have fundamental period

$$
2\pi.
$$

More generally,

$$
\sin(n x),
\qquad
\cos(n x)
$$

complete $n$ oscillations on an interval of length $2\pi$.

The functions

$$
\cos x,\sin x,\cos2x,\sin2x,\ldots
$$

are called **harmonics**.

In [ ]:
x = np.linspace(-np.pi, np.pi, 1200)

plt.figure()
for n in [1, 2, 3, 4]:
    plt.plot(x, np.sin(n*x), label=fr"$\sin({n}x)$")
plt.xlabel("$x$")
plt.ylabel("amplitude")
plt.title("The first four sine harmonics")
plt.legend()
plt.show()

### What changes as $n$ increases?

The amplitude of $\sin(nx)$ remains between $-1$ and $1$, but the frequency increases.

On the interval

$$
[-\pi,\pi],
$$

the function

$$
\sin(nx)
$$

completes exactly $n$ full oscillations.

Fourier series do not change the basic building blocks. They change the **weights** attached to those building blocks.

## 2A. Function Inner Products and the Projection Idea

Before deriving the Fourier coefficient formulas, it helps to make the vector analogy precise.

For ordinary vectors, the dot product measures how strongly one vector points in the direction of another. If

$$
\mathbf v_1,\mathbf v_2,\ldots
$$

are mutually orthogonal, then the coefficient of $\mathbf v_n$ in an expansion of a vector $\mathbf u$ is obtained by projection:

$$
\boxed{
c_n
=
\frac{
\mathbf u\cdot\mathbf v_n
}{
\mathbf v_n\cdot\mathbf v_n
}.
}
$$

For functions on an interval $[a,b]$, define the inner product by

$$
\boxed{
\langle f,g\rangle
=
\int_a^b f(x)g(x)\,dx.
}
$$

The corresponding norm is

$$
\boxed{
\|f\|
=
\sqrt{\langle f,f\rangle}
=
\sqrt{
\int_a^b f(x)^2\,dx
}.
}
$$

Two functions are orthogonal on $[a,b]$ when

$$
\boxed{
\langle f,g\rangle=0.
}
$$

If an orthogonal family

$$
\phi_0,\phi_1,\phi_2,\ldots
$$

is used to represent a function,

$$
f(x)
\sim
c_0\phi_0(x)+c_1\phi_1(x)+c_2\phi_2(x)+\cdots,
$$

then the coefficient of $\phi_n$ is obtained by the same projection principle:

$$
\boxed{
c_n
=
\frac{
\langle f,\phi_n\rangle
}{
\langle\phi_n,\phi_n\rangle
}.
}
$$

For Fourier series, the basis functions are

$$
1,\quad
\cos x,\quad
\sin x,\quad
\cos2x,\quad
\sin2x,\ldots
$$

so the familiar Fourier coefficient formulas are projection formulas in a function space.

The conceptual correspondence is

$$
\boxed{
\begin{array}{c|c}
\text{Vectors} & \text{Functions}\\
\hline
\mathbf u\cdot\mathbf v
&
\displaystyle\int_a^b f(x)g(x)\,dx
\\[3mm]
\mathbf u\perp\mathbf v
&
\displaystyle\int_a^b f(x)g(x)\,dx=0
\\[3mm]
\text{vector component}
&
\text{Fourier coefficient}
\end{array}
}
$$

## 3. Orthogonality: Why the Coefficients Can Be Isolated

The coefficient formulas are not arbitrary. They come from the orthogonality of sines and cosines.

For positive integers $m$ and $n$,

$$
\int_{-\pi}^{\pi}
\cos(mx)\cos(nx)\,dx
=
\begin{cases}
0, & m\ne n,\\
\pi, & m=n,
\end{cases}
$$

and

$$
\int_{-\pi}^{\pi}
\sin(mx)\sin(nx)\,dx
=
\begin{cases}
0, & m\ne n,\\
\pi, & m=n.
\end{cases}
$$

Also,

$$
\int_{-\pi}^{\pi}
\sin(mx)\cos(nx)\,dx
=
0.
$$

Finally,

$$
\int_{-\pi}^{\pi}\sin(nx)\,dx=0,
\qquad
\int_{-\pi}^{\pi}\cos(nx)\,dx=0.
$$

These relations are analogous to perpendicular vectors having dot product zero.

The integral

$$
\int_{-\pi}^{\pi}f(x)g(x)\,dx
$$

plays the role of a continuous inner product.

In [ ]:
x = np.linspace(-np.pi, np.pi, 2000)

product_different = np.sin(2*x)*np.sin(3*x)
product_same = np.sin(2*x)*np.sin(2*x)

plt.figure()
plt.plot(x, product_different)
plt.axhline(0)
plt.xlabel("$x$")
plt.ylabel(r"$\sin(2x)\sin(3x)$")
plt.title("Different harmonics: positive and negative areas cancel")
plt.show()

plt.figure()
plt.plot(x, product_same)
plt.axhline(0)
plt.xlabel("$x$")
plt.ylabel(r"$\sin^2(2x)$")
plt.title("Same harmonic: the integral is positive")
plt.show()

print("Numerical integral for sin(2x)sin(3x):",
      np.trapz(product_different, x))
print("Numerical integral for sin^2(2x):",
      np.trapz(product_same, x))

## 4. Deriving the Fourier Coefficients

Assume

$$
f(x)
=
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
\left[
a_n\cos(nx)+b_n\sin(nx)
\right].
$$

To isolate $a_m$, multiply both sides by

$$
\cos(mx)
$$

and integrate from $-\pi$ to $\pi$:

$$
\int_{-\pi}^{\pi}
f(x)\cos(mx)\,dx
=
\int_{-\pi}^{\pi}
\frac{a_0}{2}\cos(mx)\,dx
+
\sum_{n=1}^{\infty}
a_n
\int_{-\pi}^{\pi}
\cos(nx)\cos(mx)\,dx
$$

$$
+
\sum_{n=1}^{\infty}
b_n
\int_{-\pi}^{\pi}
\sin(nx)\cos(mx)\,dx.
$$

Orthogonality makes almost every term disappear.

The constant term contributes zero, every sine-cosine product contributes zero, and every cosine term with $n\ne m$ contributes zero.

Only the term with

$$
n=m
$$

survives:

$$
\int_{-\pi}^{\pi}
f(x)\cos(mx)\,dx
=
a_m
\int_{-\pi}^{\pi}
\cos^2(mx)\,dx.
$$

Since

$$
\int_{-\pi}^{\pi}\cos^2(mx)\,dx=\pi,
$$

we obtain

$$
\boxed{
a_m
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}
f(x)\cos(mx)\,dx.
}
$$

The sine coefficient follows in exactly the same way. Multiply by

$$
\sin(mx)
$$

and integrate:

$$
\boxed{
b_m
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}
f(x)\sin(mx)\,dx.
}
$$

To find the constant coefficient, integrate the Fourier series without multiplying by a trigonometric function:

$$
\int_{-\pi}^{\pi}f(x)\,dx
=
\frac{a_0}{2}(2\pi).
$$

Therefore,

$$
\boxed{
a_0
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}f(x)\,dx.
}
$$

Thus the coefficient formulas are

$$
\boxed{
a_0
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}f(x)\,dx,
}
$$

$$
\boxed{
a_n
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}f(x)\cos(nx)\,dx,
}
$$

and

$$
\boxed{
b_n
=
\frac{1}{\pi}
\int_{-\pi}^{\pi}f(x)\sin(nx)\,dx.
}
$$

## 5. General Period $2L$

Not every periodic function has period $2\pi$.

Suppose the function has period

$$
2L.
$$

The natural basis becomes

$$
\cos\left(\frac{n\pi x}{L}\right),
\qquad
\sin\left(\frac{n\pi x}{L}\right).
$$

Then

$$
\boxed{
f(x)
\sim
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
\left[
a_n\cos\left(\frac{n\pi x}{L}\right)
+
b_n\sin\left(\frac{n\pi x}{L}\right)
\right],
}
$$

where

$$
\boxed{
a_0
=
\frac1L
\int_{-L}^{L}f(x)\,dx,
}
$$

$$
\boxed{
a_n
=
\frac1L
\int_{-L}^{L}
f(x)
\cos\left(\frac{n\pi x}{L}\right)\,dx,
}
$$

and

$$
\boxed{
b_n
=
\frac1L
\int_{-L}^{L}
f(x)
\sin\left(\frac{n\pi x}{L}\right)\,dx.
}
$$

For a first introduction, it is usually easiest to work with

$$
L=\pi
$$

and then scale the formulas later.

## 6. Detailed Example: The Sawtooth Function $f(x)=x$

Consider

$$
f(x)=x,
\qquad
-\pi<x<\pi,
$$

and extend it periodically with period $2\pi$.

We will compute its Fourier series from the coefficient formulas.

First,

$$
a_0
=
\frac1\pi
\int_{-\pi}^{\pi}x\,dx.
$$

The integrand is odd, so

$$
\boxed{a_0=0.}
$$

Next,

$$
a_n
=
\frac1\pi
\int_{-\pi}^{\pi}x\cos(nx)\,dx.
$$

Because

$$
x
$$

is odd and

$$
\cos(nx)
$$

is even, the product is odd. Therefore,

$$
\boxed{a_n=0.}
$$

Only the sine coefficients remain:

$$
b_n
=
\frac1\pi
\int_{-\pi}^{\pi}x\sin(nx)\,dx.
$$

Now both $x$ and $\sin(nx)$ are odd, so their product is even:

$$
b_n
=
\frac{2}{\pi}
\int_0^\pi x\sin(nx)\,dx.
$$

Use integration by parts.

Let

$$
u=x,
\qquad
dv=\sin(nx)\,dx.
$$

Then

$$
du=dx,
\qquad
v=-\frac{\cos(nx)}{n}.
$$

Therefore,

$$
\int_0^\pi x\sin(nx)\,dx
=
\left[
-\frac{x\cos(nx)}{n}
\right]_0^\pi
+
\frac1n
\int_0^\pi\cos(nx)\,dx.
$$

The second integral is zero:

$$
\int_0^\pi\cos(nx)\,dx
=
\left[
\frac{\sin(nx)}{n}
\right]_0^\pi
=
0.
$$

Thus

$$
\int_0^\pi x\sin(nx)\,dx
=
-\frac{\pi\cos(n\pi)}{n}.
$$

Since

$$
\cos(n\pi)=(-1)^n,
$$

we obtain

$$
b_n
=
\frac{2}{\pi}
\left(
-\frac{\pi(-1)^n}{n}
\right)
=
\frac{2(-1)^{n+1}}{n}.
$$

Hence

$$
\boxed{
x
\sim
2
\sum_{n=1}^{\infty}
\frac{(-1)^{n+1}}{n}\sin(nx),
\qquad
-\pi<x<\pi.
}
$$

In [ ]:
def sawtooth_explorer(N=5):
    x = np.linspace(-np.pi, np.pi, 1600)
    exact = x
    approx = sawtooth_partial_sum(x, N)

    plt.figure()
    plt.plot(x, exact, label="$f(x)=x$")
    plt.plot(x, approx, label=fr"$S_{{{N}}}(x)$")
    plt.xlabel("$x$")
    plt.ylabel("$y$")
    plt.title(f"Sawtooth Fourier approximation with N={N}")
    plt.legend()
    plt.show()

interact(
    sawtooth_explorer,
    N=IntSlider(min=1, max=40, step=1, value=5, description="$N$")
);

### Periodic Extension: What Happens Outside $(-\pi,\pi)$?

The original formula

$$
f(x)=x
$$

was specified only on

$$
-\pi<x<\pi.
$$

Its Fourier series is built from sines and cosines, and every term is $2\pi$-periodic. Therefore the sum is also $2\pi$-periodic.

So the Fourier series does **not** continue the straight line $y=x$ forever.

Instead, it repeats the segment from $-\pi$ to $\pi$ every $2\pi$ units. The result is the periodic sawtooth function.

Mathematically,

$$
\boxed{
F(x+2\pi)=F(x).
}
$$

This is called the **periodic extension** of the original function.

The jump at the endpoints occurs because the right end of one copy,

$$
x\to\pi^-,
$$

must connect to the left end of the next copy,

$$
x\to-\pi^+.
$$

For the sawtooth,

$$
f(\pi^-)=\pi,
\qquad
f(-\pi^+)=-\pi,
$$

so the periodic extension has a jump of size

$$
2\pi.
$$

In [ ]:
x_long = np.linspace(-5*np.pi, 5*np.pi, 4000)
periodic = sawtooth_periodic(x_long)

plt.figure()
plt.plot(x_long, periodic, linewidth=2, label="periodic extension")

# Highlight the interval on which the original function was specified.
mask = (x_long >= -np.pi) & (x_long <= np.pi)
plt.plot(
    x_long[mask],
    periodic[mask],
    linewidth=4,
    label="original interval $(-\\pi,\\pi)$"
)

plt.axvline(-np.pi, linestyle="--", linewidth=1)
plt.axvline(np.pi, linestyle="--", linewidth=1)
plt.xlabel("$x$")
plt.ylabel("$F(x)$")
plt.title("Periodic extension of $f(x)=x$ on $(-\\pi,\\pi)$")
plt.legend()
plt.show()

### What should students notice?

As $N$ increases:

- the partial sum becomes increasingly accurate away from the endpoints;
- more high-frequency oscillations are added;
- the approximation develops a sharp transition near the periodic jump from $\pi$ back to $-\pi$.

The approximation is not produced by moving one curve toward another. It is produced by adding more harmonic components.

In [ ]:
n = np.arange(1, 16)
bn = 2*((-1)**(n+1))/n

plt.figure()
plt.stem(n, np.abs(bn))
plt.xlabel("harmonic number $n$")
plt.ylabel(r"$|b_n|$")
plt.title("Magnitude spectrum for the sawtooth Fourier series")
plt.show()

The coefficient magnitudes decay like

$$
\frac1n.
$$

This relatively slow decay is connected to the jump discontinuity in the periodic extension.

Later we will compare this with a smoother function whose coefficients decay like

$$
\frac1{n^2}.
$$

## 7. Animation: Building a Fourier Series One Harmonic at a Time

The following movie starts with the first harmonic and adds one additional sine harmonic at each step.

This is useful in class because students can literally watch the approximation being assembled from simple oscillations.

In [ ]:
def animate_sawtooth_build(max_N=20):
    x = np.linspace(-np.pi, np.pi, 1200)
    exact = x

    fig, ax = plt.subplots()
    ax.plot(x, exact, label="$f(x)=x$")
    partial_line, = ax.plot([], [], label="partial sum")
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(-4, 4)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.legend()

    def update(frame):
        N = frame + 1
        partial_line.set_data(x, sawtooth_partial_sum(x, N))
        ax.set_title(f"Sawtooth Fourier series: first {N} harmonics")
        return partial_line,

    anim = FuncAnimation(
        fig,
        update,
        frames=max_N,
        interval=350,
        blit=False
    )
    plt.close(fig)
    return display_animation(anim)

animate_sawtooth_build(18)

## Section 8.1 Summary

For a $2\pi$-periodic function,

$$
f(x)
\sim
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
\left[
a_n\cos(nx)+b_n\sin(nx)
\right].
$$

The coefficients are projections onto orthogonal trigonometric basis functions:

$$
a_0
=
\frac1\pi
\int_{-\pi}^{\pi}f(x)\,dx,
$$

$$
a_n
=
\frac1\pi
\int_{-\pi}^{\pi}f(x)\cos(nx)\,dx,
$$

$$
b_n
=
\frac1\pi
\int_{-\pi}^{\pi}f(x)\sin(nx)\,dx.
$$

The central conceptual chain is

$$
\boxed{
\text{orthogonality}
\longrightarrow
\text{coefficient formulas}
\longrightarrow
\text{harmonic decomposition}.
}
$$

# Section 8.2: Symmetry, Piecewise Functions, Convergence, and Gibbs Phenomenon

## 1. Why Symmetry Matters

Many Fourier integrals disappear automatically when the function has symmetry.

Recall:

- $f$ is **even** if

$$
f(-x)=f(x);
$$

- $f$ is **odd** if

$$
f(-x)=-f(x).
$$

The parity rules are:

$$
\text{even}\times\text{even}=\text{even},
$$

$$
\text{odd}\times\text{odd}=\text{even},
$$

$$
\text{even}\times\text{odd}=\text{odd}.
$$

Also,

$$
\int_{-L}^{L}
\text{odd function}\,dx=0.
$$

### Even functions

If $f$ is even, then

$$
f(x)\sin(nx)
$$

is odd. Therefore,

$$
\boxed{b_n=0.}
$$

The Fourier series contains only cosines:

$$
\boxed{
f(x)
\sim
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}a_n\cos(nx).
}
$$

Moreover,

$$
a_0
=
\frac{2}{\pi}
\int_0^\pi f(x)\,dx,
$$

and

$$
a_n
=
\frac{2}{\pi}
\int_0^\pi f(x)\cos(nx)\,dx.
$$

### Odd functions

If $f$ is odd, then

$$
a_0=0
$$

and

$$
a_n=0.
$$

Only sine terms remain:

$$
\boxed{
f(x)
\sim
\sum_{n=1}^{\infty}b_n\sin(nx),
}
$$

with

$$
b_n
=
\frac{2}{\pi}
\int_0^\pi f(x)\sin(nx)\,dx.
$$

## 2. Detailed Even Example: $f(x)=|x|$

Consider

$$
f(x)=|x|,
\qquad
-\pi\le x\le\pi.
$$

Because $|x|$ is even,

$$
b_n=0.
$$

First compute

$$
a_0
=
\frac{2}{\pi}
\int_0^\pi x\,dx.
$$

Therefore,

$$
a_0
=
\frac{2}{\pi}
\left[
\frac{x^2}{2}
\right]_0^\pi
=
\pi.
$$

Hence the constant term is

$$
\frac{a_0}{2}
=
\frac{\pi}{2}.
$$

Now compute

$$
a_n
=
\frac{2}{\pi}
\int_0^\pi x\cos(nx)\,dx.
$$

Integrate by parts:

$$
u=x,
\qquad
dv=\cos(nx)\,dx,
$$

so

$$
du=dx,
\qquad
v=\frac{\sin(nx)}{n}.
$$

Then

$$
\int_0^\pi x\cos(nx)\,dx
=
\left[
\frac{x\sin(nx)}{n}
\right]_0^\pi
-
\frac1n
\int_0^\pi\sin(nx)\,dx.
$$

The boundary term is zero because

$$
\sin(n\pi)=0.
$$

Also,

$$
\int_0^\pi\sin(nx)\,dx
=
\left[
-\frac{\cos(nx)}{n}
\right]_0^\pi
=
\frac{1-(-1)^n}{n}.
$$

Therefore,

$$
a_n
=
-\frac{2}{\pi n^2}
\left[
1-(-1)^n
\right].
$$

If $n$ is even,

$$
a_n=0.
$$

If $n$ is odd,

$$
a_n=-\frac{4}{\pi n^2}.
$$

Thus

$$
\boxed{
|x|
=
\frac{\pi}{2}
-
\frac{4}{\pi}
\left(
\cos x
+
\frac{\cos3x}{3^2}
+
\frac{\cos5x}{5^2}
+\cdots
\right).
}
$$

In [ ]:
def absx_explorer(N=4):
    x = np.linspace(-np.pi, np.pi, 1600)
    exact = np.abs(x)
    approx = absx_partial_sum(x, N)

    plt.figure()
    plt.plot(x, exact, label="$|x|$")
    plt.plot(x, approx, label=f"{N} nonzero cosine terms")
    plt.xlabel("$x$")
    plt.ylabel("$y$")
    plt.title("Fourier approximation of $|x|$")
    plt.legend()
    plt.show()

interact(
    absx_explorer,
    N=IntSlider(min=1, max=20, step=1, value=4, description="$N$")
);

### Comparing coefficient decay

For the sawtooth example,

$$
|b_n|
\sim
\frac1n.
$$

For $|x|$,

$$
|a_n|
\sim
\frac1{n^2}
$$

for the nonzero coefficients.

The smoother function has faster coefficient decay.

This is an important general theme:

$$
\boxed{
\text{greater smoothness}
\quad\Longrightarrow\quad
\text{faster Fourier coefficient decay}.
}
$$

In [ ]:
n = np.arange(1, 20, 2)
saw_mag = 2/n
abs_mag = 4/(np.pi*n**2)

plt.figure()
plt.plot(n, saw_mag, marker="o", label=r"sawtooth: proportional to $1/n$")
plt.plot(n, abs_mag, marker="s", label=r"$|x|$: proportional to $1/n^2$")
plt.xlabel("odd harmonic number $n$")
plt.ylabel("coefficient magnitude")
plt.title("Coefficient decay and smoothness")
plt.legend()
plt.show()

## 3. Half-Range Expansions

Sometimes a function is given only on

$$
0<x<L.
$$

To build a Fourier series, we can extend it to

$$
-L<x<L
$$

in more than one way.

### Odd extension: Fourier sine series

If we extend the function oddly, the resulting Fourier series contains only sine terms:

$$
\boxed{
f(x)
\sim
\sum_{n=1}^{\infty}
b_n
\sin\left(\frac{n\pi x}{L}\right),
}
$$

where

$$
\boxed{
b_n
=
\frac{2}{L}
\int_0^L
f(x)
\sin\left(\frac{n\pi x}{L}\right)\,dx.
}
$$

### Even extension: Fourier cosine series

If we extend the function evenly, the resulting Fourier series contains only cosine terms:

$$
\boxed{
f(x)
\sim
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
a_n
\cos\left(\frac{n\pi x}{L}\right),
}
$$

where

$$
\boxed{
a_n
=
\frac{2}{L}
\int_0^L
f(x)
\cos\left(\frac{n\pi x}{L}\right)\,dx.
}
$$

These are called **half-range expansions**.

### Example: $f(x)=x$ on $0<x<\pi$

If we choose the odd extension, we recover the sawtooth series:

$$
\boxed{
x
\sim
2
\sum_{n=1}^{\infty}
\frac{(-1)^{n+1}}{n}\sin(nx),
\qquad
0<x<\pi.
}
$$

If instead we choose the even extension, then the extension is $|x|$, so

$$
\boxed{
x
\sim
\frac{\pi}{2}
-
\frac{4}{\pi}
\left(
\cos x
+
\frac{\cos3x}{3^2}
+
\frac{\cos5x}{5^2}
+\cdots
\right),
\qquad
0<x<\pi.
}
$$

Both series equal the same original function on

$$
0<x<\pi,
$$

but they represent different periodic extensions outside that interval.

This distinction becomes important in boundary-value problems and partial differential equations.

In [ ]:
x_full = np.linspace(-np.pi, np.pi, 1200)

odd_extension = x_full
even_extension = np.abs(x_full)

plt.figure()
plt.plot(x_full, odd_extension)
plt.axvline(0)
plt.xlabel("$x$")
plt.ylabel("$f$")
plt.title("Odd extension of $f(x)=x$ from $(0,\pi)$")
plt.show()

plt.figure()
plt.plot(x_full, even_extension)
plt.axvline(0)
plt.xlabel("$x$")
plt.ylabel("$f$")
plt.title("Even extension of $f(x)=x$ from $(0,\pi)$")
plt.show()

## 4. A Piecewise Example: The Square Wave

Define

$$
f(x)
=
\begin{cases}
-1, & -\pi<x<0,\\
1, & 0<x<\pi,
\end{cases}
$$

and extend periodically.

This function is odd, so

$$
a_0=0,
\qquad
a_n=0.
$$

Thus only the sine coefficients remain:

$$
b_n
=
\frac{2}{\pi}
\int_0^\pi
\sin(nx)\,dx.
$$

Evaluate:

$$
b_n
=
\frac{2}{\pi}
\left[
-\frac{\cos(nx)}{n}
\right]_0^\pi
=
\frac{2}{\pi n}
\left[
1-(-1)^n
\right].
$$

Therefore,

$$
b_n
=
\begin{cases}
\dfrac{4}{\pi n}, & n\text{ odd},\\[2mm]
0, & n\text{ even}.
\end{cases}
$$

Hence

$$
\boxed{
f(x)
\sim
\frac4\pi
\left(
\sin x
+
\frac{\sin3x}{3}
+
\frac{\sin5x}{5}
+\cdots
\right).
}
$$

In [ ]:
def square_wave_explorer(N=5):
    x = np.linspace(-np.pi, np.pi, 2000)
    approx = square_partial_sum(x, N)

    plt.figure()
    plt.plot(x, square_wave(x), label="square wave")
    plt.plot(x, approx, label=f"{N} odd harmonics")
    plt.xlabel("$x$")
    plt.ylabel("$f(x)$")
    plt.title("Fourier approximation of a square wave")
    plt.ylim(-1.6, 1.6)
    plt.legend()
    plt.show()

interact(
    square_wave_explorer,
    N=IntSlider(min=1, max=40, step=1, value=5, description="$N$")
);

## 4A. Piecewise Continuity: Why Discontinuous Signals Are Still Allowed

Many important periodic inputs are not continuous everywhere. Square waves and sawtooth waves, for example, contain jump discontinuities.

A function is **piecewise continuous** on a finite interval if

- it is continuous except at finitely many points;
- the one-sided limits at those exceptional points are finite.

Thus a jump discontinuity is allowed.

For example, the square wave

$$
f(x)
=
\begin{cases}
-1, & -\pi<x<0,\\
1, & 0<x<\pi
\end{cases}
$$

is piecewise continuous even though it jumps at

$$
x=0.
$$

A useful sufficient condition for the standard Fourier convergence theorem is that both

$$
f
\qquad\text{and}\qquad
f'
$$

are piecewise continuous on the interval under consideration.

Under this hypothesis, the Fourier series converges to

$$
\boxed{f(x)}
$$

at a point where $f$ is continuous, and to

$$
\boxed{
\frac{f(x^-)+f(x^+)}{2}
}
$$

at a jump discontinuity.

This explains why discontinuous periodic forcing functions can still be analyzed effectively using Fourier series.

## 5. What Does the Fourier Series Converge To?

A useful practical version of the Fourier convergence theorem is:

If a periodic function is piecewise smooth, then its Fourier series converges at a point $x$ to

$$
\boxed{
\frac{f(x^-)+f(x^+)}{2}.
}
$$

Here

$$
f(x^-)
$$

is the left-hand limit and

$$
f(x^+)
$$

is the right-hand limit.

### At a point of continuity

If

$$
f(x^-)=f(x^+)=f(x),
$$

then

$$
\boxed{
S(x)=f(x).
}
$$

### At a jump discontinuity

If the one-sided limits differ, then

$$
\boxed{
S(x)
=
\frac{f(x^-)+f(x^+)}{2}.
}
$$

For the square wave at

$$
x=0,
$$

we have

$$
f(0^-)=-1,
\qquad
f(0^+)=1.
$$

Therefore,

$$
\boxed{
S(0)=0.
}
$$

## 6. Gibbs Phenomenon

Near a jump discontinuity, partial Fourier sums develop oscillatory overshoots and undershoots.

Increasing the number of terms does two things:

1. the oscillations become concentrated into a narrower region around the jump;
2. the maximum overshoot does **not** disappear.

For a jump of size

$$
J,
$$

the limiting overshoot is approximately

$$
0.08949J.
$$

For the square wave, the jump size is

$$
J=2,
$$

so the overshoot is approximately

$$
0.17898.
$$

Thus the partial sums rise to about

$$
1.179
$$

near a jump from $-1$ to $1$.

This persistent overshoot is called the **Gibbs phenomenon**.

In [ ]:
x = np.linspace(-0.8, 0.8, 2200)

plt.figure()
for N in [2, 5, 15, 40]:
    plt.plot(x, square_partial_sum(x, N), label=f"{N} odd harmonics")
plt.axhline(1)
plt.axhline(-1)
plt.axvline(0)
plt.xlabel("$x$")
plt.ylabel("$S_N(x)$")
plt.title("Gibbs phenomenon near the jump at $x=0$")
plt.legend()
plt.show()

### Animation: the oscillations narrow but the overshoot remains

The movie below zooms in near the discontinuity while increasing the number of retained harmonics.

In [ ]:
def animate_gibbs(max_N=35):
    x = np.linspace(-0.8, 0.8, 1800)

    fig, ax = plt.subplots()
    partial_line, = ax.plot([], [])
    ax.axhline(1)
    ax.axhline(-1)
    ax.axvline(0)
    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$S_N(x)$")

    def update(frame):
        N = frame + 1
        partial_line.set_data(x, square_partial_sum(x, N))
        ax.set_title(f"Gibbs phenomenon: {N} odd harmonics")
        return partial_line,

    anim = FuncAnimation(
        fig,
        update,
        frames=max_N,
        interval=220,
        blit=False
    )
    plt.close(fig)
    return display_animation(anim)

animate_gibbs(30)

## 7. A Frequency-Spectrum View of the Square Wave

The square wave contains only odd harmonics:

$$
1,3,5,7,\ldots
$$

and their amplitudes are

$$
\frac4\pi,
\quad
\frac4{3\pi},
\quad
\frac4{5\pi},
\quad
\frac4{7\pi},
\ldots
$$

This gives another way to think about a periodic function:

- the time-domain graph tells us how the signal changes with time;
- the Fourier coefficients tell us how much of each frequency is present.

In [ ]:
odd_n = np.arange(1, 20, 2)
amp = 4/(np.pi*odd_n)

plt.figure()
plt.stem(odd_n, amp)
plt.xlabel("harmonic number $n$")
plt.ylabel("sine coefficient $b_n$")
plt.title("Frequency spectrum of the square wave")
plt.show()

## Section 8.2 Summary

The most useful shortcuts and convergence facts are:

$$
\boxed{
\text{even function}
\Rightarrow
\text{cosine series only},
}
$$

$$
\boxed{
\text{odd function}
\Rightarrow
\text{sine series only}.
}
$$

For piecewise smooth periodic functions,

$$
\boxed{
S(x)
=
\frac{f(x^-)+f(x^+)}{2}.
}
$$

At continuous points this equals $f(x)$. At jumps it gives the midpoint.

Near jumps, partial sums exhibit the Gibbs phenomenon.

Half-range expansions allow a function defined only on

$$
0<x<L
$$

to be represented either by a sine series or by a cosine series, depending on whether we choose an odd or even extension.

# Section 8.3: Fourier Series as Periodic Forcing in Differential Equations

## 1. Why Fourier Series Belong in a Differential Equations Course

Consider a damped spring-mass system:

$$
\boxed{
my''+cy'+ky=F(t).
}
$$

If the forcing is a single sinusoid,

$$
F(t)=F_0\sin(\Omega t),
$$

the method of undetermined coefficients gives a sinusoidal steady-state response.

But suppose the forcing is a square wave.

A square wave is not a single sinusoid, but its Fourier series is

$$
F(t)
=
\frac{4F_0}{\pi}
\left[
\sin(\omega t)
+
\frac13\sin(3\omega t)
+
\frac15\sin(5\omega t)
+\cdots
\right].
$$

Because the differential equation is linear, we can solve separately for the response to each harmonic and then add the responses.

Thus Fourier series convert a complicated periodic forcing problem into a sequence of familiar sinusoidal forcing problems.

## 2. Response to One Harmonic

Start with

$$
my''+cy'+ky
=
F\sin(\Omega t).
$$

Seek a particular solution of the form

$$
\boxed{
y_p
=
A\sin(\Omega t)
+
B\cos(\Omega t).
}
$$

Then

$$
y_p'
=
A\Omega\cos(\Omega t)
-
B\Omega\sin(\Omega t),
$$

and

$$
y_p''
=
-A\Omega^2\sin(\Omega t)
-
B\Omega^2\cos(\Omega t).
$$

Substitute into the differential equation.

The coefficient of

$$
\sin(\Omega t)
$$

is

$$
(k-m\Omega^2)A-c\Omega B,
$$

while the coefficient of

$$
\cos(\Omega t)
$$

is

$$
c\Omega A+(k-m\Omega^2)B.
$$

Matching coefficients gives

$$
\boxed{
(k-m\Omega^2)A-c\Omega B=F,
}
$$

$$
\boxed{
c\Omega A+(k-m\Omega^2)B=0.
}
$$

Solving the two equations gives

$$
\boxed{
A
=
\frac{
F(k-m\Omega^2)
}{
(k-m\Omega^2)^2+c^2\Omega^2
},
}
$$

and

$$
\boxed{
B
=
-\frac{
Fc\Omega
}{
(k-m\Omega^2)^2+c^2\Omega^2
}.
}
$$

The amplitude of the steady-state response is

$$
R
=
\sqrt{A^2+B^2}.
$$

After simplification,

$$
\boxed{
R(\Omega)
=
\frac{
F
}{
\sqrt{
(k-m\Omega^2)^2+c^2\Omega^2
}
}.
}
$$

This is the frequency-response amplitude.

A forcing frequency near the natural frequency can produce a much larger response than other frequencies.

In [ ]:
def frequency_response(m=1.0, c=0.6, k=9.0):
    Omega = np.linspace(0.05, 8, 1600)
    R = 1/np.sqrt((k-m*Omega**2)**2 + (c*Omega)**2)

    plt.figure()
    plt.plot(Omega, R)
    plt.axvline(np.sqrt(k/m), linestyle="--",
                label=r"$\sqrt{k/m}$")
    plt.xlabel(r"forcing frequency $\Omega$")
    plt.ylabel("steady-state amplitude for unit forcing")
    plt.title("Frequency response of a damped oscillator")
    plt.legend()
    plt.show()

interact(
    frequency_response,
    m=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description="$m$"),
    c=FloatSlider(min=0.1, max=3.0, step=0.1, value=0.6, description="$c$"),
    k=FloatSlider(min=1.0, max=20.0, step=0.5, value=9.0, description="$k$")
);

## 3. Square-Wave Forcing

Let the applied force alternate between

$$
+F_0
$$

and

$$
-F_0
$$

with fundamental angular frequency

$$
\omega.
$$

Its Fourier series is

$$
\boxed{
F(t)
=
\frac{4F_0}{\pi}
\sum_{j=0}^{\infty}
\frac{
\sin[(2j+1)\omega t]
}{
2j+1
}.
}
$$

The $j$th nonzero harmonic has frequency

$$
\Omega_j=(2j+1)\omega
$$

and forcing amplitude

$$
F_j
=
\frac{4F_0}{\pi(2j+1)}.
$$

For each harmonic, compute

$$
A_j
=
\frac{
F_j(k-m\Omega_j^2)
}{
(k-m\Omega_j^2)^2+c^2\Omega_j^2
},
$$

and

$$
B_j
=
-\frac{
F_jc\Omega_j
}{
(k-m\Omega_j^2)^2+c^2\Omega_j^2
}.
$$

Then the steady-state approximation is

$$
\boxed{
y_{\mathrm{ss}}(t)
\approx
\sum_{j=0}^{N-1}
\left[
A_j\sin(\Omega_j t)
+
B_j\cos(\Omega_j t)
\right].
}
$$

In [ ]:
def square_forcing_series(t, N=8, F0=1.0, omega=1.0):
    out = np.zeros_like(np.asarray(t, dtype=float))
    for j in range(N):
        n = 2*j + 1
        out += 4*F0/(np.pi*n) * np.sin(n*omega*t)
    return out

def oscillator_square_steady_state(t, N=8, m=1.0, c=0.6, k=9.0,
                                   F0=1.0, omega=1.0):
    y = np.zeros_like(np.asarray(t, dtype=float))
    for j in range(N):
        n = 2*j + 1
        Om = n*omega
        Fj = 4*F0/(np.pi*n)
        D = (k-m*Om**2)**2 + (c*Om)**2
        A = Fj*(k-m*Om**2)/D
        B = -Fj*c*Om/D
        y += A*np.sin(Om*t) + B*np.cos(Om*t)
    return y

In [ ]:
def forcing_explorer(N=5, omega=1.0):
    t = np.linspace(0, 4*np.pi/omega, 2000)

    plt.figure()
    plt.plot(t, np.where(np.sin(omega*t) >= 0, 1.0, -1.0),
             label="ideal square wave")
    plt.plot(t, square_forcing_series(t, N=N, omega=omega),
             label=f"{N} odd harmonics")
    plt.xlabel("$t$")
    plt.ylabel("$F(t)$")
    plt.title("Square-wave forcing and its Fourier approximation")
    plt.legend()
    plt.show()

interact(
    forcing_explorer,
    N=IntSlider(min=1, max=30, step=1, value=5, description="$N$"),
    omega=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description=r"$\omega$")
);

In [ ]:
def response_explorer(N=5, m=1.0, c=0.6, k=9.0, omega=1.0):
    t = np.linspace(0, 25, 2500)
    y = oscillator_square_steady_state(
        t, N=N, m=m, c=c, k=k, F0=1.0, omega=omega
    )

    plt.figure()
    plt.plot(t, y)
    plt.xlabel("$t$")
    plt.ylabel("$y_{ss}(t)$")
    plt.title(f"Steady-state oscillator response using {N} odd harmonics")
    plt.show()

interact(
    response_explorer,
    N=IntSlider(min=1, max=20, step=1, value=5, description="$N$"),
    m=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description="$m$"),
    c=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.6, description="$c$"),
    k=FloatSlider(min=2.0, max=20.0, step=0.5, value=9.0, description="$k$"),
    omega=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description=r"$\omega$")
);

## 4. Detailed Numerical Example: Why One Harmonic Can Dominate

Take

$$
m=1,
\qquad
c=0.6,
\qquad
k=9,
$$

and choose square-wave forcing with

$$
F_0=1,
\qquad
\omega=1.
$$

The undamped natural frequency is

$$
\omega_0
=
\sqrt{\frac{k}{m}}
=
3.
$$

The square wave contains the frequencies

$$
1,3,5,7,\ldots
$$

Therefore its **third harmonic** occurs exactly at

$$
\Omega=3,
$$

which is the natural frequency of the corresponding undamped oscillator.

Damping prevents unbounded resonance, but the third harmonic can still be strongly amplified.

In [ ]:
m, c, k, F0, omega = 1.0, 0.6, 9.0, 1.0, 1.0

print("harmonic   forcing amplitude   response amplitude")
for n in [1, 3, 5, 7]:
    Fj = 4*F0/(np.pi*n)
    Om = n*omega
    D = (k-m*Om**2)**2 + (c*Om)**2
    response_amp = Fj/np.sqrt(D)
    print(f"{n:>4d}       {Fj:>10.6f}          {response_amp:>10.6f}")

For these parameters, the forcing amplitudes decrease with harmonic number, but the third harmonic response is larger than the first harmonic response.

This is a crucial differential-equations insight:

$$
\boxed{
\text{large response}
\ne
\text{largest forcing coefficient}.
}
$$

The differential equation itself acts as a **frequency filter**.

A harmonic close to resonance can matter more than a lower-frequency harmonic with a larger Fourier coefficient.

In [ ]:
nvals = np.arange(1, 16, 2)
forcing_amp = 4/(np.pi*nvals)

response_amp = []
for n in nvals:
    Om = n
    D = (9-Om**2)**2 + (0.6*Om)**2
    response_amp.append((4/(np.pi*n))/np.sqrt(D))
response_amp = np.asarray(response_amp)

plt.figure()
plt.stem(nvals, forcing_amp)
plt.xlabel("odd harmonic $n$")
plt.ylabel("forcing amplitude")
plt.title("Square-wave forcing spectrum")
plt.show()

plt.figure()
plt.stem(nvals, response_amp)
plt.xlabel("odd harmonic $n$")
plt.ylabel("oscillator response amplitude")
plt.title("The oscillator filters the Fourier harmonics")
plt.show()

## 5. Comparing Fourier Steady State with Direct Numerical Simulation

The Fourier construction gives the steady-state response.

A direct numerical solution of the differential equation also contains a transient determined by the initial conditions.

For

$$
my''+cy'+ky=F(t),
$$

the complete response can be viewed as

$$
\boxed{
y(t)
=
y_{\text{transient}}(t)
+
y_{\text{steady state}}(t).
}
$$

With positive damping,

$$
c>0,
$$

the transient decays, and the numerical solution approaches the periodic Fourier steady state.

In [ ]:
def compare_numeric_and_fourier(m=1.0, c=0.6, k=9.0,
                                F0=1.0, omega=1.0, N=12):
    def rhs(t, z):
        y, v = z
        force = F0 if np.sin(omega*t) >= 0 else -F0
        return [v, (force-c*v-k*y)/m]

    t_eval = np.linspace(0, 60, 6000)
    sol = solve_ivp(
        rhs,
        (0, 60),
        [0.0, 0.0],
        t_eval=t_eval,
        rtol=1e-8,
        atol=1e-10,
        max_step=0.02
    )

    y_fourier = oscillator_square_steady_state(
        t_eval, N=N, m=m, c=c, k=k, F0=F0, omega=omega
    )

    mask = t_eval >= 40

    plt.figure()
    plt.plot(t_eval[mask], sol.y[0, mask], label="direct numerical solution")
    plt.plot(t_eval[mask], y_fourier[mask], linestyle="--",
             label=f"Fourier steady state, {N} odd harmonics")
    plt.xlabel("$t$")
    plt.ylabel("$y(t)$")
    plt.title("After the transient decays: numerical and Fourier responses")
    plt.legend()
    plt.show()

compare_numeric_and_fourier()

The two curves agree increasingly well when:

- enough Fourier harmonics are retained;
- the transient has had time to decay;
- the numerical integration resolves the switching times accurately.

The small discrepancies near sharp forcing transitions are related to truncating the Fourier series and to the fact that the ideal square wave is discontinuous.

## 6. Animation: How the Differential Equation Filters Added Harmonics

The next movie adds square-wave harmonics one at a time and displays the corresponding steady-state oscillator response.

Notice that adding a harmonic to the forcing does **not** add the same amount to the displacement. Each frequency is weighted by the dynamics of the differential equation.

In [ ]:
def animate_filtered_response(max_N=12, m=1.0, c=0.6, k=9.0, omega=1.0):
    t = np.linspace(0, 20, 1800)

    fig, ax = plt.subplots()
    line, = ax.plot([], [])
    ax.set_xlim(0, 20)
    ax.set_ylim(-0.55, 0.55)
    ax.set_xlabel("$t$")
    ax.set_ylabel("$y_{ss}(t)$")

    def update(frame):
        N = frame + 1
        y = oscillator_square_steady_state(
            t, N=N, m=m, c=c, k=k, F0=1.0, omega=omega
        )
        line.set_data(t, y)
        ax.set_title(f"Oscillator response using {N} odd harmonics")
        return line,

    anim = FuncAnimation(
        fig,
        update,
        frames=max_N,
        interval=350,
        blit=False
    )
    plt.close(fig)
    return display_animation(anim)

animate_filtered_response(12)

## 7. Conceptual Bridge to Methods Already Learned

Fourier series do not replace the methods from Chapter 4.

They organize them.

For example, suppose

$$
F(t)
=
F_1\sin(\omega t)
+
F_3\sin(3\omega t)
+
F_5\sin(5\omega t)
+\cdots.
$$

By linearity,

$$
L[y]
=
F(t)
$$

can be separated into

$$
L[y_1]
=
F_1\sin(\omega t),
$$

$$
L[y_3]
=
F_3\sin(3\omega t),
$$

$$
L[y_5]
=
F_5\sin(5\omega t),
$$

and so on.

Each equation can be solved by a familiar method such as undetermined coefficients.

Then

$$
\boxed{
y
=
y_1+y_3+y_5+\cdots.
}
$$

So the conceptual chain is

$$
\boxed{
\text{periodic forcing}
\longrightarrow
\text{Fourier decomposition}
\longrightarrow
\text{solve harmonic ODEs}
\longrightarrow
\text{superpose the responses}.
}
$$

## 8. Suggested Classroom Checkpoint

Consider

$$
y''+4y=f(t),
$$

where $f$ is the $2\pi$-periodic square wave

$$
f(t)
=
\begin{cases}
1, & 0<t<\pi,\\
-1, & -\pi<t<0.
\end{cases}
$$

### Questions

1. Write the first three nonzero Fourier harmonics of $f(t)$.
2. Which forcing harmonic is closest to the natural frequency of the undamped oscillator?
3. Why should that harmonic receive special attention?
4. If the system were damped, what qualitative change would you expect in the resonant response?

### Instructor discussion

The square wave begins with

$$
f(t)
\approx
\frac4\pi
\left[
\sin t
+
\frac13\sin3t
+
\frac15\sin5t
\right].
$$

The homogeneous equation

$$
y''+4y=0
$$

has natural frequency

$$
\omega_0=2.
$$

None of the odd harmonics is exactly equal to $2$. The first harmonic has frequency $1$ and the third harmonic has frequency $3$, both one unit away.

This example is useful because it reinforces that resonance is determined by a comparison between the system's natural frequency and the forcing frequencies, not merely by the size of the Fourier coefficients.

## 9. Optional Looking Ahead: Where Do These Orthogonal Functions Come From?

For this course, we use sine and cosine functions because they form an orthogonal family that is especially convenient for periodic functions.

There is also a deeper differential-equations explanation.

Consider the boundary-value problem

$$
y''+\lambda y=0,
$$

with

$$
y(0)=0,
\qquad
y(L)=0.
$$

Nonzero solutions occur only for special parameter values

$$
\boxed{
\lambda_n
=
\left(
\frac{n\pi}{L}
\right)^2,
\qquad
n=1,2,3,\ldots
}
$$

and the corresponding solutions are

$$
\boxed{
y_n(x)
=
\sin\left(
\frac{n\pi x}{L}
\right).
}
$$

These special solutions are called **eigenfunctions**, and they are mutually orthogonal on $[0,L]$.

Thus the sine functions used in Fourier series also arise naturally as solutions of boundary-value problems.

A systematic theory explaining this connection is called **Sturm--Liouville theory**.

We will not develop Sturm--Liouville theory in this chapter. The important idea for now is simply:

$$
\boxed{
\text{Fourier basis functions}
\quad\text{also arise naturally from differential equations.}
}
$$

This connection becomes especially important in later courses on partial differential equations and mathematical physics.

# Chapter Summary

Fourier series provide a bridge between periodic functions and linear differential equations.

The chapter can be summarized by four ideas.

## 1. Harmonic decomposition

A periodic function can be represented by

$$
\boxed{
\frac{a_0}{2}
+
\sum_{n=1}^{\infty}
\left[
a_n\cos(nx)+b_n\sin(nx)
\right].
}
$$

## 2. Orthogonality determines the coefficients

$$
\boxed{
a_n
=
\frac1\pi
\int_{-\pi}^{\pi}
f(x)\cos(nx)\,dx,
}
$$

$$
\boxed{
b_n
=
\frac1\pi
\int_{-\pi}^{\pi}
f(x)\sin(nx)\,dx.
}
$$

## 3. Symmetry saves work

Even functions produce cosine series; odd functions produce sine series.

## 4. Linear ODEs respond harmonic by harmonic

If

$$
L[y]=F(t)
$$

and $F(t)$ is decomposed into a Fourier series, linearity allows us to solve for each harmonic response and add the results.

The broad modeling idea is

$$
\boxed{
\text{complicated periodic input}
\longrightarrow
\text{simple harmonic inputs}
\longrightarrow
\text{simple harmonic responses}
\longrightarrow
\text{total response}.
}
$$